# Bio-IFN: Bio-Inspired Interactive Feedback Network

**Bio-Inspired Deep Learning Model**

**Bio-Inspiration**: Ventral stream-based visual pathways  
**Deep Learning Enhancement**: Feedback and interactive encoding  
**Improvement Area**: Accurate edge extraction in noisy environments

Ventral stream (the "what" pathway) with feedback mechanisms.

In [ ]:
from pathlib import Path
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'opencv-python', 'numpy', 'tqdm', 'scikit-learn'], check=False)
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, cv2
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score

OUTPUT_DIR = Path('..') / 'bio DL' / 'outputs' / 'Bio-IFN'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Bio-IFN with Interactive Feedback

In [ ]:
class InteractiveFeedback(nn.Module):
    """Interactive feedback mechanism"""
    def __init__(self, channels):
        super().__init__()
        self.forward_path = nn.Conv2d(channels, channels, 3, padding=1)
        self.feedback_path = nn.Conv2d(channels, channels, 3, padding=1)
        self.gate = nn.Conv2d(channels * 2, channels, 1)
    def forward(self, x, feedback=None):
        forward_feat = F.relu(self.forward_path(x))
        if feedback is not None:
            feedback_feat = F.relu(self.feedback_path(feedback))
            gated = torch.sigmoid(self.gate(torch.cat([forward_feat, feedback_feat], dim=1)))
            return forward_feat * gated + feedback_feat * (1 - gated)
        return forward_feat

class BioIFN(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Conv2d(3, 64, 3, padding=1)
        self.ifb1 = InteractiveFeedback(64)
        self.enc2 = nn.Conv2d(64, 128, 3, padding=1)
        self.ifb2 = InteractiveFeedback(128)
        self.enc3 = nn.Conv2d(128, 256, 3, padding=1)
        # Feedback connections
        self.fb3to2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.fb2to1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.edge = nn.Conv2d(64, 1, 1)
    def forward(self, x):
        h, w = x.shape[2:]
        # Forward pass
        e1 = F.relu(self.enc1(x))
        e2 = F.relu(self.enc2(F.max_pool2d(e1, 2)))
        e3 = F.relu(self.enc3(F.max_pool2d(e2, 2)))
        # Interactive feedback
        fb2 = self.fb3to2(e3)
        e2_refined = self.ifb2(e2, fb2)
        fb1 = self.fb2to1(e2_refined)
        e1_refined = self.ifb1(e1, fb1)
        return torch.sigmoid(self.edge(e1_refined))

model = BioIFN().to(DEVICE).eval()
print(f"✓ Bio-IFN: {sum(p.numel() for p in model.parameters()):,} params")

In [ ]:
class EdgeDataset(Dataset):
    def __init__(self, root, split='test'):
        self.img_dir, self.gt_dir = root / split / 'images', root / split / 'edges'
        self.images = sorted(list(self.img_dir.glob('*.jpg')) + list(self.img_dir.glob('*.png')))[:20]
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img_path = self.images[idx]
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        gt = cv2.imread(str(self.gt_dir / img_path.name.replace('.jpg', '.png')), 0)
        gt = gt.astype(np.float32) / 255.0 if gt is not None else np.zeros(img.shape[:2], dtype=np.float32)
        return torch.from_numpy(img.transpose(2, 0, 1)), torch.from_numpy(gt), img_path.name

loader = DataLoader(EdgeDataset(Path('..') / 'datasets' / 'HED_Small', 'test'), batch_size=1)
preds, gts = [], []
with torch.no_grad():
    for imgs, gt, _ in tqdm(loader):
        preds.extend([model(imgs.to(DEVICE))[i,0].cpu().numpy() for i in range(imgs.shape[0])])
        gts.extend([gt[i].cpu().numpy() for i in range(gt.shape[0])])

def metrics(preds, labels):
    t, ois, ap, al = np.linspace(0.05, 0.95, 30), [], [], []
    for p, l in zip(preds, labels):
        l = cv2.dilate((l>0.5).astype(np.float32), np.ones((3,3))).flatten()
        p = cv2.GaussianBlur(p, (3,3), 0).flatten()
        ap.append(p); al.append(l)
        ois.append(max([2*np.sum((p>=th)*l)/(2*np.sum((p>=th)*l)+np.sum((p>=th)*(1-l))+np.sum((p<th)*l)+1e-8) for th in t]))
    fp, fl = np.concatenate(ap), np.concatenate(al)
    ods = max([(2*np.sum((fp>=th)*fl)/(2*np.sum((fp>=th)*fl)+np.sum((fp>=th)*(1-fl))+np.sum((fp<th)*fl)+1e-8), th) for th in t])
    return {'ODS': ods[0], 'ODS_thresh': ods[1], 'OIS': np.mean(ois), 'AP': average_precision_score(fl, fp) if np.sum(fl)>0 else 0}

m = metrics(preds, gts)
print(f"\nBio-IFN: ODS={m['ODS']:.4f} | OIS={m['OIS']:.4f} | AP={m['AP']:.4f}")

import json
with open(OUTPUT_DIR / 'bioifn_metrics.json', 'w') as f:
    json.dump({'model': 'Bio-IFN', 'bio': 'Ventral stream + Interactive feedback', 'improvement': 'Noise robustness', 'metrics': m}, f, indent=2)
print("✅ Bio-IFN complete!")